In [1]:
import matplotlib.pyplot as plt
import numpy as np
import os
import sys
import pickle

path = os.getcwd().split(os.sep + 'GUI')[0]
if path not in sys.path:
    print("not here")
    sys.path.append(path)

from neurolib.models.aln import ALNModel
from neurolib.utils import plotFunctions as plotFunc
from neurolib.utils import costFunctions as cost
import neurolib.dashboard.functions as functions
import neurolib.dashboard.data as data
    
# This will reload all imports as soon as the code changes
%load_ext autoreload
%autoreload 2 

#path = os.path.join(os.getcwd(), "plots")

not here


In [2]:
# read case
print(os.getcwd())
case = os.getcwd().split(os.sep)[-1]
print(case)

/mnt/antares_raid/home/salfenmoser/neurolib/GUI/current/gui/data/00101
00101


### Bistability

In [3]:
aln = ALNModel()
N = aln.params.N

data.set_parameters(aln)

state_vars = aln.state_vars
init_vars = aln.init_vars

##############################################################
def setinit(init_vars_, model):
    state_vars = model.state_vars
    init_vars = model.init_vars
    for iv in range(len(init_vars)):
        for sv in range(len(state_vars)):
            if state_vars[sv] in init_vars[iv]:
                #print("set init vars ", )
                if model.params[init_vars[iv]].ndim == 2:
                    model.params[init_vars[iv]][0,:] = init_vars_[sv]
                else:
                    model.params[init_vars[iv]][0] = init_vars_[sv]
                    
##############################################################               
def setmaxmincontrol(max_c_c, min_c_c, max_c_r, min_c_r):
    import numpy as np
    
    max_cntrl = np.zeros(( 6 ))
    min_cntrl = np.zeros(( 6 ))
    
    max_cntrl[0] = max_c_c
    min_cntrl[0] = min_c_c
    max_cntrl[1] = max_c_c
    min_cntrl[1] = min_c_c
    max_cntrl[2] = max_c_r
    min_cntrl[2] = min_c_r
    max_cntrl[3] = max_c_r
    min_cntrl[3] = min_c_r
    max_cntrl[4] = max_c_r
    min_cntrl[4] = min_c_r
    max_cntrl[5] = max_c_r
    min_cntrl[5] = min_c_r
            
    return max_cntrl, min_cntrl

#####################################################
def getclosest(k_, found_solution, exc, inh, already_tried_):
    import numpy as np
    if len(found_solution) == 0:
        print("no solutions found")
        return -1
    
    start_ind = -1
    for j_ in found_solution:
        if j_ not in already_tried_ and j_ != k_:
            start_ind = j_
            break
            
    if start_ind == -1:
        return -1
        
    min_dist = np.sqrt((exc[k_] - exc[start_ind])**2 + (inh[k_] - inh[start_ind])**2)
    min_i = start_ind
        
    print(found_solution, already_tried_)
        
    if len(found_solution) == len(already_tried_):
        print("already tried all options")
        min_i = -1
        return min_i
    
    for i_ in found_solution:
        if i_ not in already_tried_:
            if i_ != k_ and i_ != min_i:
                dist_ = np.sqrt((exc[k_] - exc[i_])**2 + (inh[k_] - inh[i_])**2)
                if dist_ < min_dist:
                    min_dist = dist_
                    min_i = i_
                    
    if min_i == 0 and 0 in already_tried_:
        return -1
    
    return min_i

In [4]:
##### LOAD BOUNDARIES
data_file = 'bi.pickle'
with open(data_file,'rb') as f:
    load_array= pickle.load(f)
exc = load_array[0]
inh = load_array[1]
print(len(exc))
#plt.scatter(exc, inh)

147


In [5]:
bestControl_init = [None] * len(exc)
bestState_init = [None] * len(exc)
cost_init = [None] * len(exc)
runtime_init = [None] * len(exc)
grad_init = [None] * len(exc)
phi_init = [None] * len(exc)
costnode_init = [None] * len(exc)
weights_init = [None] * len(exc)

conv_init = [[False]*2] * len(exc)

In [6]:
bestControl_0 = [None] * len(exc)
bestState_0 = [None] * len(exc)
cost_0 = [None] * len(exc)
runtime_0 = [None] * len(exc)
grad_0 = [None] * len(exc)
phi_0 = [None] * len(exc)
costnode_0 = [None] * len(exc)
weights_0 = [None] * len(exc)

conv_0 = [[False]*2] * len(exc)

In [7]:
bestControl_1 = [None] * len(exc)
bestState_1 = [None] * len(exc)
cost_1 = [None] * len(exc)
runtime_1 = [None] * len(exc)
grad_1 = [None] * len(exc)
phi_1 = [None] * len(exc)
costnode_1 = [None] * len(exc)
weights_1 = [None] * len(exc)

conv_1 = [[False]*2] * len(exc)

In [8]:
initVars = [None] * len(exc)
target = [None] * len(exc)
cost_uncontrolled = [None] * len(exc)

cgv_list = [None, "HS", "FR", "PR", "CD", "LS", "DY", "WYL", "HZ", None]

In [9]:
dur_pre = 10
dur_post = 10

n_pre = int(np.around(dur_pre/aln.params.dt + 1.,1))
n_post = int(np.around(dur_post/aln.params.dt + 1.,1))

tol = 1e-32
start_step = 10.
c_scheme = np.zeros(( 1,1 ))
c_scheme[0,0] = 1.
u_mat = np.identity(1)
u_scheme = np.array([[1.]])

c_var = [ [0], [1], [0,1]]
p_var = [ [0], [0], [0]]

### CURRENTS
cntrl_vars_0 = [0,1]
prec_vars = [0]

if case[0] == '0':    # low to high
    max_I = [3., -3.]
elif case[0] == '1':
    max_I = [-3., 3.]
    
if case[1] == '0':    # sparsity
    factor_ws = 1.
    factor_we = 0.
elif case[1] == '1':  # energy
    factor_ws = 0.
    factor_we = 1.
    
if case[3] == '0':
    cntrl_vars_init = [0]
elif case[3] == '1':
    cntrl_vars_init = [1]
elif case[3] == '2':
    cntrl_vars_init = [0,1]

if case[4] == '0':
    dur = 100
    trans_time = 0.8
elif case[4] == '1':
    dur = 400
    trans_time = 0.95
    
maxC = [5., -5., 0.18, 0.]

n_dur = int(np.around(dur/aln.params.dt + 1.,1))
max_cntrl, min_cntrl = setmaxmincontrol(maxC[0], maxC[1], maxC[2], maxC[3])

In [10]:
init_file = 'control_init_' + case + '.pickle'
final_file = 'control_' + case + '.pickle'
case_1 = case[0] + case[1] + '0' + case[3] + case[4]
final_file_1 = 'control_' + case_1 + '.pickle'

In [11]:
if os.path.isfile(init_file) :
    print("file found")
    
    with open(init_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_init = load_array[0]
    bestState_init = load_array[1]
    cost_init = load_array[2]
    runtime_init = load_array[3]
    grad_init = load_array[4]
    phi_init = load_array[5]
    costnode_init = load_array[6]
    weights_init = load_array[7]

file found


In [12]:
# get initial parameters and target states

i_stepsize = 5
i_range = range(0, len(exc),i_stepsize)
i_range_0 = range(0, len(exc),i_stepsize)
i_range_1 = range(0, len(exc),i_stepsize)
data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = 3000.
    
    control0 = aln.getZeroControl()
    control0 = functions.step_control(aln, maxI_ = max_I[0])

    aln.run(control=control0)
    
    target_rates = np.zeros((2))
    target_rates[0] = aln.rates_exc[0,-1] 
    target_rates[1] = aln.rates_inh[0,-1]

    control0 = functions.step_control(aln, maxI_ = max_I[1])
    aln.run(control=control0)

    init_state_vars = np.zeros(( len(state_vars) ))
    for j in range(len(state_vars)):
        if aln.state[state_vars[j]].size == 1:
            init_state_vars[j] = aln.state[state_vars[j]][0]
        else:
            init_state_vars[j] = aln.state[state_vars[j]][0,-1]

    initVars[i] = init_state_vars
    
    aln.params.duration = dur

    target[i] = aln.getZeroTarget()
    target[i][:,0,:] = target_rates[0]
    target[i][:,1,:] = target_rates[1]

-------  0 0.4000000000000001 0.3500000000000001
-------  5 0.4000000000000001 0.40000000000000013
-------  10 0.4250000000000001 0.42500000000000016
-------  15 0.4500000000000001 0.4500000000000002
-------  20 0.4500000000000001 0.4750000000000002
-------  25 0.4250000000000001 0.5000000000000002
-------  30 0.4250000000000001 0.5250000000000002
-------  35 0.5500000000000003 0.5250000000000002
-------  40 0.5250000000000001 0.5500000000000003
-------  45 0.5000000000000002 0.5750000000000003
-------  50 0.47500000000000014 0.6000000000000003
-------  55 0.4250000000000001 0.6250000000000003
-------  60 0.5500000000000003 0.6250000000000003
-------  65 0.5000000000000002 0.6500000000000004
-------  70 0.4500000000000001 0.6750000000000004
-------  75 0.5750000000000002 0.6750000000000004
-------  80 0.5250000000000001 0.7000000000000004
-------  85 0.47500000000000014 0.7250000000000004
-------  90 0.6000000000000003 0.7250000000000004
-------  95 0.5250000000000001 0.750000000000000

In [13]:
# get uncontrolled cost

data.set_parameters(aln)

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
    cost.setParams(1.0, 0.0, 0.0)

##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    control0 = aln.getZeroControl()

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = 0

    bestControl_init_, bestState_init_, cost_init_, runtime_init_, grad_init_, phi_init_, costnode_init_ = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    cost_uncontrolled[i] = cost_init_[0]

-------  0 0.4000000000000001 0.3500000000000001
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5902.406479238383
Gradient descend method:  None
RUN  0 , total integrated cost =  5902.406479238383
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5097.289828199723
Gradient descend method:  None
RUN  0 , total integrated cost =  5097.289828199723
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  10 0.4250000000000001 0.42500000000000016
set cost params:  1.0 0.0 0.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9111.456490210901
Gradient descend method:  None
RUN  0 , total integrated cost =  9111.456490210901
Improved over  0  iterations in  0.0  seconds by  0.0  percent.
-------  15 0.4500000000000001 0.4500000000000002

In [14]:
factor_iteration = 20.

for i in i_range:
    print("------- ", i, exc[i], inh[i])
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.
    
    aln.params.duration = dur
        
##### zero control as input for uncontrolled cost
    setinit(initVars[i], aln)
    
    if not type(bestControl_init[i]) == type(None):
        continue
        
    control0 = aln.getZeroControl()

    ##### initial guess
    weight_ = 10
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(100 * factor_iteration)

    weights_init[i] = cost.getParams()

    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
    
    j = 1
    while cost_init[i][-j] == 0.:
        j += 1
    
    weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
    print("weight = ", weight_)
    cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

    setinit(initVars[i], aln)
    control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

    # "HS", "FR", "PR", "HZ"
    cgv = None
    max_it = int(500 * factor_iteration)

    weights_init[i] = cost.getParams()
    
    bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
        control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
        startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
        t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
        prec_variables_ = prec_vars, transition_time_ = trans_time)
        
    with open(init_file,'wb') as f:
        pickle.dump([bestControl_init, bestState_init, cost_init, runtime_init, grad_init, phi_init,
                 costnode_init, weights_init], f)

-------  0 0.4000000000000001 0.3500000000000001
-------  5 0.4000000000000001 0.40000000000000013
-------  10 0.4250000000000001 0.42500000000000016
-------  15 0.4500000000000001 0.4500000000000002
-------  20 0.4500000000000001 0.4750000000000002
-------  25 0.4250000000000001 0.5000000000000002
-------  30 0.4250000000000001 0.5250000000000002
-------  35 0.5500000000000003 0.5250000000000002
-------  40 0.5250000000000001 0.5500000000000003
-------  45 0.5000000000000002 0.5750000000000003
-------  50 0.47500000000000014 0.6000000000000003
-------  55 0.4250000000000001 0.6250000000000003
-------  60 0.5500000000000003 0.6250000000000003
-------  65 0.5000000000000002 0.6500000000000004
-------  70 0.4500000000000001 0.6750000000000004
-------  75 0.5750000000000002 0.6750000000000004
-------  80 0.5250000000000001 0.7000000000000004
-------  85 0.47500000000000014 0.7250000000000004
-------  90 0.6000000000000003 0.7250000000000004
-------  95 0.5250000000000001 0.750000000000000

In [15]:
"""
#plot initial guesses
for i in i_range:
    print("---------", i)
        
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i]],
        [costnode_init[i]], [weights_init[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
    plt.show()
"""

'\n#plot initial guesses\nfor i in i_range:\n    print("---------", i)\n        \n    aln.params.mue_ext_mean = exc[i] * 5.\n    aln.params.mui_ext_mean = inh[i] * 5.\n\n    plotFunc.plot_control_current(aln, [bestControl_init[i]],\n        [costnode_init[i]], [weights_init[i]], dur,\n        dur_pre, dur_post, initVars[i], target[i], \'\', filename_ = \'\', transition_time_ = trans_time,\n        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)\n    plt.show()\n'

In [16]:
found_solution = []
no_solution = []
factor_iteration = 20.
already_tried = [ [] for _ in range(len(exc)) ]

for k in range(len(i_range)**2):
    print('------------------------------------------------------------')
    print('--------------------', k)
    print('------------------------------------------------------------')
        
    print("found solution: ", found_solution)
    print("no solution: ", no_solution)
    
    if len(i_range) == len(found_solution) + len(no_solution):
        print("found solution for all parameters")
        break


    for i in i_range:
        print("------- ", i, exc[i], inh[i])        

        if np.abs(np.mean(bestState_init[i][0,0,-300:]) - target[i][0,0,-1]) < 0.1 * np.abs(
            np.mean(bestState_init[i][0,0,-100:]) - bestState_init[i][0,0,0]) and np.abs(
            np.mean(bestState_init[i][0,1,-300:]) - target[i][0,1,-1]) < 0.1 * np.abs(
            np.mean(bestState_init[i][0,1,-100:]) - bestState_init[i][0,1,0]) and np.amax(
            bestState_init[i][0,0,:]) < target[i][0,0,-1] + 1. and np.amax(
            bestState_init[i][0,1,:]) < target[i][0,1,-1]:
            # and np.amin(bestState_init[i][0,0,:]) > bestState_init[i][0,0,0] - 1.
            #and np.amin(bestState_init[i][0,1,:]) > bestState_init[i][0,1,0] - 1.:
            if i not in found_solution:
                print("found solution for ", i)
                found_solution.append(i)
            if i in no_solution:
                no_solution.pop(no_solution.index(i))
            continue

        if len(found_solution) == 0:
            continue
            
        closest_ = getclosest(i, found_solution, exc, inh, already_tried[i])
        print("closest index ", closest_)

        weight_ = 10
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
            
        if i != 0 and closest_ != -1:
            control0 = bestControl_init[closest_][:,:,n_pre-1:-n_post+1]
            if closest_ not in already_tried[i]:
                already_tried[i].append(closest_)
                        
        if closest_ == -1:
            print("all options tried already")
            if i not in no_solution:
                no_solution.append(i)
                continue

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(100 * factor_iteration)

        weights_init[i] = cost.getParams()
        
        print("precision vars = ", prec_vars)

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

        j = 1
        while cost_init[i][-j] == 0.:
            j += 1

        weight_ = 10 * cost_uncontrolled[i] / cost_init[i][-j]
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int(500 * factor_iteration)

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        with open(init_file,'wb') as f:
            pickle.dump([bestControl_init, bestState_init, cost_init, runtime_init, grad_init, phi_init,
                         costnode_init, weights_init], f)

------------------------------------------------------------
-------------------- 0
------------------------------------------------------------
found solution:  []
no solution:  []
-------  0 0.4000000000000001 0.3500000000000001
found solution for  0
-------  5 0.4000000000000001 0.40000000000000013
found solution for  5
-------  10 0.4250000000000001 0.42500000000000016
found solution for  10
-------  15 0.4500000000000001 0.4500000000000002
found solution for  15
-------  20 0.4500000000000001 0.4750000000000002
[0, 5, 10, 15] []
closest index  15
set cost params:  1.0 0.0 10.0
precision vars =  [0]
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  12767.39223606153
Gradient descend method:  None
RUN  1 , total integrated cost =  12743.852174274589
RUN  2 , total integrated cost =  12738.409479404672
RUN  3 , total integrated cost =  12738.129391459866
RUN  4 , total integrated cost =  12738.116671836447
RUN  5 , total integrated cost =  12738.116454255902
RUN

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  1342 , total integrated cost =  54.45789161098727
Improved over  1342  iterations in  89.4566213786602  seconds by  99.34085403023889  percent.
Problem in initial value trasfer:  Vmean_exc -56.63980151116903 -56.63980155235615
weight =  1511.6096084423677
set cost params:  1.0 0.0 1511.6096084423677
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  8231.73886712348
Gradient descend method:  None
RUN  1 , total integrated cost =  8229.263317908975
RUN  2 , total integrated cost =  8229.2546584496
RUN  3 , total integrated cost =  8229.251890150657
RUN  4 , total integrated cost =  8229.195898126673
RUN  5 , total integrated cost =  8229.163842602626
RUN  6 , total integrated cost =  8229.161222242872
RUN  7 , total integrated cost =  8229.102539370988
RUN  8 , total integrated cost =  8229.07200018313
RUN  9 , total integrated cost =  8229.07021016893
RUN  10 , total integrated cost =  8229.004281946576
RUN  11 , total integrat

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  74 , total integrated cost =  8227.483624977818
Improved over  74  iterations in  4.96545054204762  seconds by  0.051693113865127316  percent.
Problem in initial value trasfer:  Vmean_exc -56.63987123600673 -56.639869610247395
-------  30 0.4250000000000001 0.5250000000000002
[0, 5, 10, 15] []
closest index  15
set cost params:  1.0 0.0 10.0
precision vars =  [0]
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  8008.302120186098
Gradient descend method:  None
RUN  1 , total integrated cost =  7979.578072590945
RUN  2 , total integrated cost =  128.04587514302682
RUN  3 , total integrated cost =  69.53356461054302
RUN  4 , total integrated cost =  61.97066860318408
RUN  5 , total integrated cost =  61.14101818349031
RUN  6 , total integrated cost =  60.751735923581066
RUN  7 , total integrated cost =  60.55938934637798
RUN  8 , total integrated cost =  60.4251310035308
RUN  9 , total integrated cost =  60.35408434805712
RUN  1

...

RUN  2 , total integrated cost =  85.80090772176773
RUN  3 , total integrated cost =  85.02581454525833
RUN  4 , total integrated cost =  84.69451572764606
RUN  5 , total integrated cost =  84.49881058017642
RUN  6 , total integrated cost =  84.36224768738037
RUN  7 , total integrated cost =  84.25460184542314
RUN  8 , total integrated cost =  84.1618473321351
RUN  9 , total integrated cost =  84.07581120082334
RUN  10 , total integrated cost =  83.99472257661363
RUN  11 , total integrated cost =  83.91626546890986
RUN  12 , total integrated cost =  83.83940500673454
RUN  13 , total integrated cost =  83.7633938689184
RUN  14 , total integrated cost =  83.68781275116405
RUN  15 , total integrated cost =  83.60977145850678
RUN  16 , total integrated cost =  83.52874000480809
RUN  17 , total integrated cost =  83.44798392128098
RUN  18 , total integrated cost =  83.37134274450233
RUN  19 , total integrated cost =  83.29730321479781
RUN  20 , total integrated cost =  83.22437718413293
RUN

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  892 , total integrated cost =  73.11207713965955
Improved over  892  iterations in  58.78280019760132  seconds by  37.389250866797966  percent.
Problem in initial value trasfer:  Vmean_exc -56.631598010623264 -56.63159804032254
weight =  972.878030036668
set cost params:  1.0 0.0 972.878030036668
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  7112.868784888847
Gradient descend method:  None
RUN  1 , total integrated cost =  7112.494440972554
RUN  2 , total integrated cost =  7112.493790919982
RUN  3 , total integrated cost =  7112.493677114022
RUN  4 , total integrated cost =  7112.4861928995115
RUN  5 , total integrated cost =  7112.47516370113
RUN  6 , total integrated cost =  7112.4749685691695
RUN  7 , total integrated cost =  7112.474851465187
RUN  8 , total integrated cost =  7112.37222906705
RUN  9 , total integrated cost =  7112.300538734863
RUN  10 , total integrated cost =  7112.300535164286
RUN  11 , total integr

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  22 , total integrated cost =  7112.300534352667
Improved over  22  iterations in  1.5761163346469402  seconds by  0.007989048488951767  percent.
Problem in initial value trasfer:  Vmean_exc -56.631594320808674 -56.63159384777133
-------  60 0.5500000000000003 0.6250000000000003
-------  65 0.5000000000000002 0.6500000000000004
-------  70 0.4500000000000001 0.6750000000000004
-------  75 0.5750000000000002 0.6750000000000004
-------  80 0.5250000000000001 0.7000000000000004
-------  85 0.47500000000000014 0.7250000000000004
-------  90 0.6000000000000003 0.7250000000000004
-------  95 0.5250000000000001 0.7500000000000004
-------  100 0.4500000000000001 0.7750000000000005
-------  105 0.5750000000000002 0.7750000000000005
-------  110 0.5000000000000002 0.8000000000000005
-------  115 0.4250000000000001 0.8250000000000005
-------  120 0.5500000000000003 0.8250000000000005
-------  125 0.47500000000000014 0.8500000000000005
-------  130 0.6000000000

In [17]:
factor_iteration = 20
full_converge = False
conv_init = [[False]*2] * len(exc)

for i in range(len(conv_init)):
    if i not in i_range:
        conv_init[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    print("------------------------------------------------")
    print('-------------------------', counter)
    
    if counter > 20:
        break
        
    print(conv_init[::i_stepsize])
    full_converge = True
    
    for conv in conv_init[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range:        

        print("------- ", i, exc[i], inh[i])
        
        if conv_init[i] == [True, True]:
            continue
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        j = 1
        while cost_init[i][-j] == 0.:
            j += 1
                       
        weight_ = (factor_we * weights_init[i][1] * cost_uncontrolled[i] / cost_init[i][-j]
                   + factor_ws * weights_init[i][2] * cost_uncontrolled[i] / cost_init[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        setinit(initVars[i], aln)
        control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1]

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_init[i] = cost.getParams()

        bestControl_init[i], bestState_init[i], cost_init[i], runtime_init[i], grad_init[i], phi_init[i], costnode_init[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        with open(init_file,'wb') as f:
            pickle.dump([bestControl_init, bestState_init, cost_init, runtime_init, grad_init, phi_init,
                         costnode_init, weights_init], f)
            
        if j == cost_init[i].shape[0]-1:
            print("converged for ", i)
            if conv_init[i][0]:
                conv_init[i] = [True, True]
            else:
                conv_init[i] = [True, False]
            continue
    
        print("no convergence")
            
    counter += 1

------------------------------------------------
------------------------- 0
[[False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False]]
-------  0 0.4000000000000001 0.3500000000000001
weight =  4565.061323953685
set cost params:  1.0 0.0 4565.061323953685
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5898.954457173973
Gradient descend method:  None
RUN  1 , total integrated cost =  5898.886461640382
RUN  2 , total integrated cost =  5898.885037695302
RUN  3 , total integrated cost =  5898.884914083593
RUN  4 , total integrated cost =  5898.88488

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  23 , total integrated cost =  5898.884875241609
Improved over  23  iterations in  1.5795983988791704  seconds by  0.0011795638171179235  percent.
Problem in initial value trasfer:  Vmean_exc -56.6272719488988 -56.62727365897679
no convergence
-------  5 0.4000000000000001 0.40000000000000013
weight =  1623.1091826613238
set cost params:  1.0 0.0 1623.1091826613238
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5093.819595618996
Gradient descend method:  None
RUN  1 , total integrated cost =  5093.8155851134925
RUN  2 , total integrated cost =  5093.8153785124305
RUN  3 , total integrated cost =  5093.815339930951
RUN  4 , total integrated cost =  5093.815329254355
RUN  5 , total integrated cost =  5093.815325442442
RUN  6 , total integrated cost =  5093.815323659138
RUN  7 , total integrated cost =  5093.8153226726945
RUN  8 , total integrated cost =  5093.815322075234
RUN  9 , total integrated cost =  5093.81532169967
RUN 

ERROR:root:Problem in initial value trasfer


Control only changes marginally.
RUN  105 , total integrated cost =  5093.65532885049
Improved over  105  iterations in  6.704977685585618  seconds by  0.003224825014342514  percent.
Problem in initial value trasfer:  Vmean_exc -56.62446074695043 -56.62446070660722
no convergence
-------  10 0.4250000000000001 0.42500000000000016
weight =  2901.695726769684
set cost params:  1.0 0.0 2901.695726769684
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9107.117658560657
Gradient descend method:  None
RUN  1 , total integrated cost =  9107.080015262089
RUN  2 , total integrated cost =  9107.079599355358
RUN  3 , total integrated cost =  9107.079534718761
RUN  4 , total integrated cost =  9107.079518682447
RUN  5 , total integrated cost =  9107.079513940536
RUN  6 , total integrated cost =  9107.079512342092
RUN  7 , total integrated cost =  9107.079511652051
RUN  8 , total integrated cost =  9107.079511243111
RUN  9 , total integrated cost =  9107.079510986905
RUN  10

...

RUN  1 , total integrated cost =  33889.01899309702
Control only changes marginally.
RUN  1 , total integrated cost =  33889.01899309702
Improved over  1  iterations in  0.19049324095249176  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.703343736758484 -56.70334372373667
converged for  105
-------  110 0.5000000000000002 0.8000000000000005
-------  115 0.4250000000000001 0.8250000000000005
-------  120 0.5500000000000003 0.8250000000000005
-------  125 0.47500000000000014 0.8500000000000005
-------  130 0.6000000000000003 0.8500000000000005
-------  135 0.5250000000000001 0.8750000000000006
-------  140 0.4500000000000001 0.9000000000000006
-------  145 0.5750000000000002 0.9000000000000006
------------------------------------------------
------------------------- 5
[[True, True], [True, True], [True, False], [True, True], [True, False], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, Tr

ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  9108.317953951862
Control only changes marginally.
RUN  1 , total integrated cost =  9108.317953951862
Improved over  1  iterations in  0.18352519161999226  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.64646847630789 -56.64646854995276
converged for  10
-------  15 0.4500000000000001 0.4500000000000002
-------  20 0.4500000000000001 0.4750000000000002
weight =  3361.8844164465
set cost params:  1.0 0.0 3361.8844164465
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  12734.328595896022
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  12734.328595896022
Control only changes marginally.
RUN  1 , total integrated cost =  12734.328595896022
Improved over  1  iterations in  0.18881497532129288  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.668922830138676 -56.66892761777642
converged for  20
-------  25 0.4250000000000001 0.5000000000000002
-------  30 0.4250000000000001 0.5250000000000002
-------  35 0.5500000000000003 0.5250000000000002
-------  40 0.5250000000000001 0.5500000000000003
-------  45 0.5000000000000002 0.5750000000000003
-------  50 0.47500000000000014 0.6000000000000003
-------  55 0.4250000000000001 0.6250000000000003
-------  60 0.5500000000000003 0.6250000000000003
-------  65 0.5000000000000002 0.6500000000000004
-------  70 0.4500000000000001 0.6750000000000004
-------  75 0.5750000000000002 0.6750000000000004
-------  80 0.5250000000000001 0.7000000000000004
-------  85 0.47500000000000014 0.7250000000000004
-------  90 0.6000000000000

In [18]:
if os.path.isfile(final_file) :
    print("file found")
    
    with open(final_file,'rb') as f:
        load_array = pickle.load(f)

    bestControl_0 = load_array[0]
    bestState_0 = load_array[1]
    cost_0 = load_array[2]
    runtime_0 = load_array[3]
    grad_0 = load_array[4]
    phi_0 = load_array[5]
    costnode_0 = load_array[6]
    weights_0 = load_array[7]

file found


In [19]:
factor_iteration = 20
conv_0 = [[False]*2] * len(exc)
full_converge = False

for i in range(len(conv_0)):
    if i not in i_range_0:
        conv_0[i] = [True, True]

counter = 0

while full_converge == False:
    print('---------------', counter)
    
    if counter > 20:
        break
    
    print(conv_0[::i_stepsize])
    full_converge = True
    
    for conv in conv_0[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break
        
    counter += 1
    
    for i in i_range_0:
        print("------- ", i, exc[i], inh[i])
        
        if conv_0[i] == [True, True]:
            continue
            
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.

    # exc and inh control current 

        setinit(initVars[i], aln)
        aln.params.duration = dur

        if not type(bestControl_0[i]) == type(None):
            control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1]
        else:
            control0 = bestControl_init[i][:,:,n_pre-1:-n_post+1].copy()
            weights_0[i] = weights_init[i]
            cost_0[i] = cost_init[i]

        cgv = None
        max_it = 500 * factor_iteration

        j = 1
        while cost_0[i][-j] == 0.:
            j += 1

        weight_ = (factor_we * weights_0[i][1] * cost_uncontrolled[i] / cost_0[i][-j]
                           + factor_ws * weights_0[i][2] * cost_uncontrolled[i] / cost_0[i][-j]) - 1
        print("weight = ", weight_)
        cost.setParams(1.0, weight_ * factor_we, weight_ * factor_ws)

        weights_0[i] = cost.getParams()

        bestControl_0[i], bestState_0[i], cost_0[i], runtime_0[i], grad_0[i], phi_0[i], costnode_0[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_0,
            prec_variables_ = prec_vars, transition_time_ = trans_time)

        with open(final_file,'wb') as f:
            pickle.dump([bestControl_0, bestState_0, cost_0, runtime_0, grad_0, phi_0,
                     costnode_0, weights_0], f)
            
        if j == cost_0[i].shape[0]-1:
            print("converged for ", i)
            if conv_0[i][0]:
                conv_0[i] = [True, True]
            else:
                conv_0[i] = [True, False]
            continue
    
        print("no convergence")

--------------- 0
[[False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False]]
-------  0 0.4000000000000001 0.3500000000000001
weight =  6061.439656736909
set cost params:  1.0 0.0 6061.439656736909
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5901.432876726286
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  5901.432876726286
Control only changes marginally.
RUN  1 , total integrated cost =  5901.432876726286
Improved over  1  iterations in  0.502196729183197  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.626587585422755 -56.62659561582741
converged for  0
-------  5 0.4000000000000001 0.40000000000000013
weight =  2124.7018523410125
set cost params:  1.0 0.0 2124.7018523410125
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  5094.89189557216
Gradient descend method:  None


ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  5094.89189557216
Control only changes marginally.
RUN  1 , total integrated cost =  5094.89189557216
Improved over  1  iterations in  0.4864536002278328  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.625412507117424 -56.625394040558916
converged for  5
-------  10 0.4250000000000001 0.42500000000000016
weight =  3710.758355848221
set cost params:  1.0 0.0 3710.758355848221
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  9109.00173545138
Gradient descend method:  None


...

RUN  1 , total integrated cost =  33889.01899309702
Control only changes marginally.
RUN  1 , total integrated cost =  33889.01899309702
Improved over  1  iterations in  0.48382262513041496  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.70334373675867 -56.70334372373686
converged for  105
-------  110 0.5000000000000002 0.8000000000000005
weight =  3.999999999999999
set cost params:  1.0 0.0 3.999999999999999
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  19226.098318201533
Gradient descend method:  None
RUN  1 , total integrated cost =  19226.098318201533
Control only changes marginally.
RUN  1 , total integrated cost =  19226.098318201533
Improved over  1  iterations in  0.36238965578377247  seconds by  0.0  percent.
converged for  110
-------  115 0.4250000000000001 0.8250000000000005
weight =  3.999999999999999
set cost params:  1.0 0.0 3.999999999999999
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  584

ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  38726.23388182693
Control only changes marginally.
RUN  1 , total integrated cost =  38726.23388182693
Improved over  1  iterations in  1.4919503461569548  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.70018722822266 -56.70018721141817
converged for  130
-------  135 0.5250000000000001 0.8750000000000006
weight =  4.0
set cost params:  1.0 0.0 4.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  23532.636143093983
Gradient descend method:  None
RUN  1 , total integrated cost =  23532.636143093983
Control only changes marginally.
RUN  1 , total integrated cost =  23532.636143093983
Improved over  1  iterations in  1.177598986774683  seconds by  0.0  percent.
converged for  135
-------  140 0.4500000000000001 0.9000000000000006
weight =  4.0
set cost params:  1.0 0.0 4.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  10019.968518582271
Gradient descend method:  None
RUN  1 , tot

ERROR:root:Problem in initial value trasfer


RUN  1 , total integrated cost =  33287.53249090299
Control only changes marginally.
RUN  1 , total integrated cost =  33287.53249090299
Improved over  1  iterations in  0.9545381311327219  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.70354250496047 -56.70354247755563
converged for  145
--------------- 2
[[True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True], [True, True]]
full convergence


In [20]:
"""
for i in i_range_0:
    
    aln.params.mue_ext_mean = exc[i] * 5.
    aln.params.mui_ext_mean = inh[i] * 5.

    plotFunc.plot_control_current(aln, [bestControl_init[i], bestControl_0[i]],
        [costnode_init[i], costnode_0[i]], [weights_init[i], weights_0[i]], dur,
        dur_pre, dur_post, initVars[i], target[i], '', filename_ = '', transition_time_ = trans_time,
        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)
"""

'\nfor i in i_range_0:\n    \n    aln.params.mue_ext_mean = exc[i] * 5.\n    aln.params.mui_ext_mean = inh[i] * 5.\n\n    plotFunc.plot_control_current(aln, [bestControl_init[i], bestControl_0[i]],\n        [costnode_init[i], costnode_0[i]], [weights_init[i], weights_0[i]], dur,\n        dur_pre, dur_post, initVars[i], target[i], \'\', filename_ = \'\', transition_time_ = trans_time,\n        labels_ = ["init", "sparse control" + str(i)], print_cost_ = False)\n'

In [21]:
if os.path.isfile(final_file_1) :
    print("file found")
    
    with open(final_file_1,'rb') as f:
        load_array = pickle.load(f)

    bestControl_1 = load_array[0]
    bestState_1 = load_array[1]
    cost_1 = load_array[2]
    runtime_1 = load_array[3]
    grad_1 = load_array[4]
    phi_1 = load_array[5]
    costnode_1 = load_array[6]
    weights_1 = load_array[7]

file found


In [22]:
factor_iteration = 20
full_converge = False

for i in range(len(conv_1)):
    if i not in i_range_1:
        conv_1[i] = [True, True]
        
counter = 0

while full_converge == False:
    
    print('---------------', counter)
    if counter > 20:
        break
    
    print(conv_1[::i_stepsize])
    full_converge = True
    
    for conv in conv_1[::i_stepsize]:
        if not conv[0]:
            full_converge = False
            break
        if not conv[1]:
            full_converge = False
            break
    
    if full_converge:
        print("full convergence")
        break

    for i in i_range_1:        

        print("------- ", i, exc[i], inh[i])
        
        if conv_1[i] == [True, True]:
            continue
            
        aln.params.mue_ext_mean = exc[i] * 5.
        aln.params.mui_ext_mean = inh[i] * 5.
        
        if not type(bestControl_1[i]) == type(None):
            control0 = bestControl_1[i][:,:,n_pre-1:-n_post+1].copy()
        else:
            control0 = bestControl_0[i][:,:,n_pre-1:-n_post+1].copy()
            cost_1[i] = cost_0[i]
        
        cost.setParams(1.0, 1. * factor_we, 1. * factor_ws)

        setinit(initVars[i], aln)

        # "HS", "FR", "PR", "HZ"
        cgv = None
        max_it = int( 500 * factor_iteration )

        weights_1[i] = cost.getParams()

        bestControl_1[i], bestState_1[i], cost_1[i], runtime_1[i], grad_1[i], phi_1[i], costnode_1[i] = aln.A1(
            control0, target[i], c_scheme, u_mat, u_scheme, max_iteration_ = max_it, tolerance_ = tol,
            startStep_ = start_step, max_control_ = max_cntrl, min_control_ = min_cntrl, t_sim_ = dur,
            t_sim_pre_ = dur_pre, t_sim_post_ = dur_post, CGVar = cgv, control_variables_ = cntrl_vars_init,
            prec_variables_ = prec_vars, transition_time_ = trans_time)
        
        with open(final_file_1,'wb') as f:
            pickle.dump([bestControl_1, bestState_1, cost_1, runtime_1, grad_1, phi_1,
                 costnode_1, weights_1], f)
            
        j = 1
        while cost_1[i][-j] == 0.:
            j += 1
            
        if j == cost_1[i].shape[0]-1:
            print("converged for ", i)
            if conv_1[i][0]:
                conv_1[i] = [True, True]
            else:
                conv_1[i] = [True, False]
            continue
    
        print("no convergence")
        
    counter += 1

--------------- 0
[[False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False], [False, False]]
-------  0 0.4000000000000001 0.3500000000000001
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  1.0405383106424675
Gradient descend method:  None
RUN  1 , total integrated cost =  1.0405383106424675
Control only changes marginally.
RUN  1 , total integrated cost =  1.0405383106424675
Improved over  1  iterations in  0.1700428407639265  seconds by  0.0  percent.


ERROR:root:Problem in initial value trasfer


converged for  0
-------  5 0.4000000000000001 0.40000000000000013
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  2.477309460766982
Gradient descend method:  None
RUN  1 , total integrated cost =  2.477309460766982
Control only changes marginally.
RUN  1 , total integrated cost =  2.477309460766982
Improved over  1  iterations in  0.165648328140378  seconds by  0.0  percent.
Problem in initial value trasfer:  Vmean_exc -56.624467995185206 -56.624467861337344
converged for  5
-------  10 0.4250000000000001 0.42500000000000016
set cost params:  1.0 0.0 1.0
interpolate adjoint :  True True True
RUN  0 , total integrated cost =  2.52548901913712
Gradient descend method:  None
RUN  1 , total integrated cost =  2.52548901913712
Control only changes marginally.
RUN  1 , total integrated cost =  2.52548901913712
Improved over  1  iterations in  0.16200529038906097  seconds by  0.0  percent.
converged for  10
-------  15 0.4500000000000001 

RUN  90 , total integrated cost =  3.8580831604803967
RUN  100 , total integrated cost =  3.8580829728262347
RUN  110 , total integrated cost =  3.858082827149256
RUN  120 , total integrated cost =  3.8580826395251813
RUN  130 , total integrated cost =  3.858082493863065
RUN  140 , total integrated cost =  3.858082306269089
RUN  150 , total integrated cost =  3.858082160621807
RUN  160 , total integrated cost =  3.8580819730579656
RUN  170 , total integrated cost =  3.8580818274254858
RUN  180 , total integrated cost =  3.8580816398901185
RUN  190 , total integrated cost =  3.858081494270908
RUN  200 , total integrated cost =  3.858081306762839
RUN  300 , total integrated cost =  3.858079641801296
RUN  400 , total integrated cost =  3.8580779779627172
RUN  500 , total integrated cost =  3.858076315247226
RUN  600 , total integrated cost =  3.858074653657414
RUN  700 , total integrated cost =  3.858072993190269
RUN  800 , total integrated cost =  3.85807133384521
RUN  900 , total integr

ERROR:root:Problem in initial value trasfer


...

RUN  90 , total integrated cost =  3.854083383784468
RUN  100 , total integrated cost =  3.854082956822306
RUN  110 , total integrated cost =  3.8540816495978856
RUN  120 , total integrated cost =  3.854080596882852
RUN  130 , total integrated cost =  3.8540792910335693
RUN  140 , total integrated cost =  3.854079030088772
RUN  150 , total integrated cost =  3.854071498390167
RUN  160 , total integrated cost =  3.854071269116019
RUN  170 , total integrated cost =  3.8540703338767583
RUN  180 , total integrated cost =  3.8540695426033373
RUN  190 , total integrated cost =  3.854068827954752
RUN  200 , total integrated cost =  3.8540680146223085
RUN  300 , total integrated cost =  3.854060752220622
RUN  400 , total integrated cost =  3.8540078860786675
RUN  500 , total integrated cost =  3.8539695231162825
RUN  600 , total integrated cost =  3.853922336256854
RUN  700 , total integrated cost =  3.8538979868611793
RUN  800 , total integrated cost =  3.8538917705637945
RUN  900 , total int

RUN  3 , total integrated cost =  3.852795216245962
RUN  4 , total integrated cost =  3.8527952156786456
RUN  5 , total integrated cost =  3.852795215100373
RUN  6 , total integrated cost =  3.8527952091907616
RUN  7 , total integrated cost =  3.85279516720784
RUN  8 , total integrated cost =  3.852795161278509
RUN  9 , total integrated cost =  3.852795160699293
RUN  10 , total integrated cost =  3.852795160132923
RUN  11 , total integrated cost =  3.852795154427634
RUN  12 , total integrated cost =  3.8527951121971142
RUN  13 , total integrated cost =  3.852795104144098
RUN  14 , total integrated cost =  3.8527951035482624
RUN  15 , total integrated cost =  3.852795102996212
RUN  16 , total integrated cost =  3.8527950987404287
RUN  17 , total integrated cost =  3.8527950543415783
RUN  18 , total integrated cost =  3.8527950431665063
RUN  19 , total integrated cost =  3.8527950425492836
RUN  20 , total integrated cost =  3.85279504201211
RUN  30 , total integrated cost =  3.8527949303

RUN  1300 , total integrated cost =  3.848892259597292
RUN  1400 , total integrated cost =  3.8488905934666313
RUN  1500 , total integrated cost =  3.848888729545252
RUN  1600 , total integrated cost =  3.848884578866663
RUN  1700 , total integrated cost =  3.8488803876434514
RUN  1800 , total integrated cost =  3.848877070589536
RUN  1900 , total integrated cost =  3.848874348625588
RUN  2000 , total integrated cost =  3.848871725240301
RUN  3000 , total integrated cost =  3.8487727058280417
RUN  4000 , total integrated cost =  3.8486828219545144
RUN  5000 , total integrated cost =  3.848658339414072
RUN  6000 , total integrated cost =  3.848629077430357
RUN  7000 , total integrated cost =  3.8485403479896787
RUN  8000 , total integrated cost =  3.848376075346381
RUN  9000 , total integrated cost =  3.8482066438928433
RUN  10000 , total integrated cost =  3.848192473693909
RUN  10000 , total integrated cost =  3.848192473693909
Improved over  10000  iterations in  643.6682218704373  s

RUN  3 , total integrated cost =  3.847773056962697
RUN  4 , total integrated cost =  3.847773023255616
RUN  5 , total integrated cost =  3.847773022592388
RUN  6 , total integrated cost =  3.847773022138185
RUN  7 , total integrated cost =  3.847773019214121
RUN  8 , total integrated cost =  3.8477729779543757
RUN  9 , total integrated cost =  3.8477729658334527
RUN  10 , total integrated cost =  3.8477729653167487
RUN  11 , total integrated cost =  3.847772964769043
RUN  12 , total integrated cost =  3.847772957601706
RUN  13 , total integrated cost =  3.8477729214811505
RUN  14 , total integrated cost =  3.847772917061375
RUN  15 , total integrated cost =  3.847772916576186
RUN  16 , total integrated cost =  3.8477729159837604
RUN  17 , total integrated cost =  3.8477729013380784
RUN  18 , total integrated cost =  3.8477728563950158
RUN  19 , total integrated cost =  3.8477728535657385
RUN  20 , total integrated cost =  3.847772853116677
RUN  30 , total integrated cost =  3.84777267

KeyboardInterrupt: 